# Disjoint dbPTM holdout v1 — cohort construction

This notebook acquires and freezes a larger external cohort before any prediction. It excludes the complete prior 2,077-site external source, PLMD overlap, and proteins homologous to either source. Stop after the cohort audit; this notebook intentionally contains no inference cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Store a fine-grained read-only GitHub token in Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
!apt-get update -qq && apt-get install -y -qq mmseqs2

In [ ]:
import json

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Frozen paths and source artifacts

In [ ]:
CONFIG = Path('experiment/configs/external_dbptm_disjoint_holdout_v1.json')
ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-disjoint-holdout-v1')
ARCHIVE = ROOT / 'source/Ubiquitination.tgz'
NORMALIZED_BENCHMARK = ROOT / 'source/dbptm_ubiquitination_normalized.csv'
SOURCE_AUDIT = ROOT / 'source/source_audit.json'
SEQUENCE_CACHE = ROOT / 'uniprot_sequences.json'
COHORT_DIR = ROOT / 'cohort'
TRAINING_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
CONTEXT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
PAPER_MODEL = Path('/content/drive/MyDrive/MMUbiPred-replication/artifacts/DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5')
PRIOR_BENCHMARK = Path('replication/MMUbiPred/benchmark.csv')
PRIOR_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v1/uniprot_external_sequences.json')
PRIOR_COHORT_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v3/cohort')
ROOT.mkdir(parents=True, exist_ok=True)
for required in [CONFIG, TRAINING_SEQUENCE_CACHE, CONTEXT_RUN_DIR / 'best.pt', CONTEXT_RUN_DIR / 'refit_manifest.json', CONTEXT_RUN_DIR / 'refit_summary.json', PAPER_MODEL, PRIOR_BENCHMARK, PRIOR_SEQUENCE_CACHE, PRIOR_COHORT_DIR / 'external_cohort.tsv', PRIOR_COHORT_DIR / 'cohort_lock.json']:
    if not required.exists():
        raise FileNotFoundError(required)
print('All frozen model, training, and prior-external artifacts were found.')

## Acquire and normalize the official benchmark

The archive checksum, raw counts, duplicate identifiers, non-lysine centres, unsupported residues, and normalized CSV hash are audited before mapping.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/normalize_dbptm_benchmark.py',
    '--archive', ARCHIVE,
    '--output', NORMALIZED_BENCHMARK,
    '--audit', SOURCE_AUDIT,
])
source_audit = json.loads(SOURCE_AUDIT.read_text())
display(source_audit)

## Resolve full proteins through UniProt

This cell is resumable. The unusually large number of historical entry names is expected; canonical mappings are validated later.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/fetch_external_benchmark_sequences.py',
    '--benchmark', NORMALIZED_BENCHMARK,
    '--output', SEQUENCE_CACHE,
])

## Construct the disjoint, homology-filtered cohort

This excludes PLMD and the complete prior 2,077-site external source before MMseqs2. It writes no model predictions.

In [ ]:
LOCK_PATH = COHORT_DIR / 'cohort_lock.json'
FEASIBILITY_PATH = COHORT_DIR / 'cohort_feasibility.json'
if LOCK_PATH.exists():
    print('Existing frozen cohort found; construction was not repeated.')
else:
    try:
        run_live([
            sys.executable, '-u', 'experiment/scripts/prepare_external_benchmark.py',
            '--config', CONFIG,
            '--benchmark', NORMALIZED_BENCHMARK,
            '--external-sequence-cache', SEQUENCE_CACHE,
            '--training-sequence-cache', TRAINING_SEQUENCE_CACHE,
            '--released-model', PAPER_MODEL,
            '--context-run-dir', CONTEXT_RUN_DIR,
            '--prior-external-benchmark', PRIOR_BENCHMARK,
            '--prior-external-sequence-cache', PRIOR_SEQUENCE_CACHE,
            '--prior-external-cohort-dir', PRIOR_COHORT_DIR,
            '--output-dir', COHORT_DIR,
        ])
    except subprocess.CalledProcessError:
        if FEASIBILITY_PATH.exists():
            print('Cohort was not frozen. Pre-inference feasibility audit:')
            display(json.loads(FEASIBILITY_PATH.read_text()))
        raise

cohort_lock = json.loads(LOCK_PATH.read_text())
display({
    'status': cohort_lock['status'],
    'validation': cohort_lock['validation'],
    'exact_leakage_filter': cohort_lock['exact_leakage_filter'],
    'homology_filter': cohort_lock['homology_filter'],
    'final_cohort': cohort_lock['final_cohort'],
})

## Mandatory stop

Send the source and cohort audits for review. No prediction is authorized in this notebook, even if the cohort freezes successfully.